In [ ]:
!pip install torch torchvision timm

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.transforms import AutoAugment, AutoAugmentPolicy
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import timm
import torch.nn as nn
import torch.optim as optim
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.cuda.amp import autocast, GradScaler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder = "/content/drive/MyDrive/Dataset_2025/ImagesV2"                             #Update it based on the experiment
asv19_test_folder = "/content/drive/MyDrive/Dataset_2025/ImagesV2/test/ASV19LA"     #Update it based on the experiment
itw_test_folder =   "/content/drive/MyDrive/Dataset_2025/ImagesV2/test/In-the-wild" #Update it based on the experiment
asv21_test_folder = "/content/drive/MyDrive/Dataset_2025/ImagesV2/test/ASV21DF"     #Update it based on the experiment
asv21_val_folder = "/content/drive/MyDrive/Dataset_2025/ImagesV2/val/ASV21DF"       #Update it based on the experiment


model_folder = "/content/drive/MyDrive/Training_2025/Final_OtherModels/ResNet50/V2_MelSpectW2V/" #Update it based on the experiment

In [ ]:
# Model's parameters:
model_name = 'resnet50' #'convnext_xlarge' 'tf_efficientnet_b4_ns' <---- Update it based on the experiment
num_classes = 2
batch_size = 128
epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_size = 224



# Data transform:
transform_train = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]) # normalize spectrograms if 3-channel
])


In [ ]:
# Setting and loading the training data:
train_dataset = ImageFolder(folder + '/train', transform=transform_train)
val_dataset = ImageFolder(folder +'/val/ASV19LA', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [ ]:
# Model initialization:
model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
model.to(device)

In [ ]:
# Loss Function and Optimizer:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

In [ ]:
scaler = GradScaler()


# Early stopping and best model tracking:
best_val_acc = 0.0
best_model_path = model_folder+"/model_bestresults.pth"
early_stop_counter = 0
patience = 25  # For early stopping


# Training:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
    if (epoch+1) % 5 == 0:
      torch.save(model.state_dict(), os.path.join(model_folder, f"model_epoch{epoch+1}.pth"))
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {running_loss:.4f}")

    # Validation Loop
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for val_inputs, val_labels in val_loader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
            with autocast():
                val_outputs = model(val_inputs)
                loss = criterion(val_outputs, val_labels)
            val_loss += loss.item()
            _, predicted = torch.max(val_outputs, 1)
            total += val_labels.size(0)
            correct += (predicted == val_labels).sum().item()

    val_acc = correct / total
    print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved at epoch {epoch+1} with accuracy {best_val_acc:.4f}")
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early stopping triggered.")
            break

In [ ]:
# Confusion Matrix:
# Just for internal assessment not used in the thesis document
def save_cm_like_example(y_true, y_pred, class_names, out_path, title="Confusion Matrix"):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    n = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))  # <-- IMPORTANT (forces NxN)

    fig = plt.figure(figsize=(12, 8), dpi=160)
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(title)
    plt.colorbar()

    ticks = np.arange(n)
    plt.xticks(ticks, class_names, rotation=90)
    plt.yticks(ticks, class_names)

    # annotate
    thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(n):
        for j in range(n):
            plt.text(j, i, f"{cm[i, j]:d}",
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    # Match your example labels
    plt.xlabel("True")
    plt.ylabel("Predicted")

    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

    return cm

In [ ]:
#Validation on ASVspoof 2019 validation set:
model.eval()
val_preds = []
val_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

        val_preds.extend(preds.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())

# Metrics
val_acc = accuracy_score(val_labels, val_preds)
print(f"\nValidation Accuracy: {val_acc:.4f}")

class_names = ["bonafide", "spoof"]
print("\nValidation Classification Report:")
print(classification_report(val_labels, val_preds, target_names=class_names))


# Confusion Matrix
cm_path = model_folder+"val_results"
cm = save_cm_like_example(val_labels, val_preds, class_names, cm_path, title="Confusion Matrix")
print("Saved:", cm_path)
print(cm)

Evaluation

In [ ]:
#Reload the model:
#Since the code is working for hours in some experiments, thus we need to reload it when shutdown
MODEL_NAME = model_name
NUM_CLASSES = 2
PTH_PATH = model_folder+"model_bestresults.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES
).to(device)

state_dict = torch.load(PTH_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()


In [ ]:
#Evaluation sets calling and loading:
#Normally each set is evaluated separately to make sure the code run through
asv19_test_dataset = ImageFolder(asv19_test_folder, transform=transform)
asv21_test_dataset = ImageFolder(asv21_test_folder, transform=transform)
asv21_val_dataset = ImageFolder(asv21_val_folder, transform=transform)
itw_test_dataset = ImageFolder(itw_test_folder, transform=transform)

asv19_test_loader = DataLoader(asv19_test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
asv21_test_loader = DataLoader(asv21_test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)
asv21_val_loader = DataLoader(asv21_val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
itw_test_loader = DataLoader(itw_test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


In [ ]:
#Evaluating ASVspoof 2019 LA evalution set:
model.eval()
all_preds_asv19 = []
all_labels_asv19 = []

with torch.no_grad():
    for inputs, labels in asv19_test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

        all_preds_asv19.extend(preds.cpu().numpy())
        all_labels_asv19.extend(labels.cpu().numpy())

# Metrics
acc = accuracy_score(all_labels_asv19, all_preds_asv19)
print(f"\nAccuracy: {acc:.4f}")

class_names = ["bonafide", "spoof"]
print("\nClassification Report:")
print(classification_report(all_labels_asv19, all_preds_asv19, target_names=class_names))


# CM
cm_path = model_folder+"Test_ASV19LA"
cm = save_cm_like_example(all_labels_asv19, all_preds_asv19, class_names, cm_path, title="Confusion Matrix")
print("Saved:", cm_path)
print(cm)


In [ ]:
#Evaluating In-the-Wild dataset:

model.eval()
all_preds_itw = []
all_labels_itw = []

with torch.no_grad():
    for inputs, labels in itw_test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

        all_preds_itw.extend(preds.cpu().numpy())
        all_labels_itw.extend(labels.cpu().numpy())

# Metrics
acc = accuracy_score(all_labels_itw, all_preds_itw)
print(f"\nAccuracy: {acc:.4f}")

class_names = ["bonafide", "spoof"]
print("\nClassification Report:")
print(classification_report(all_labels_itw, all_preds_itw, target_names=class_names))


# Confusion Matrix
cm_path = model_folder+"Test_In-the-wild"
cm = save_cm_like_example(all_labels_itw, all_preds_itw, class_names, cm_path, title="Confusion Matrix")
print("Saved:", cm_path)
print(cm)

In [ ]:
#Evaluating ASVspoof 2021 DF progress set:
model.eval()
all_preds_asv21v = []
all_labels_asv21v = []

with torch.no_grad():
    for inputs, labels in asv21_val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

        all_preds_asv21v.extend(preds.cpu().numpy())
        all_labels_asv21v.extend(labels.cpu().numpy())

# Metrics
acc = accuracy_score(all_labels_asv21v, all_preds_asv21v)
print(f"\nAccuracy: {acc:.4f}")

class_names = ["bonafide", "spoof"]
print("\nClassification Report:")
print(classification_report(all_labels_asv21v, all_preds_asv21v, target_names=class_names))

# Confusion Matrix
cm_path = model_folder+"Test_ASV21DF_val"
cm = save_cm_like_example(all_labels_asv21v, all_preds_asv21v, class_names, cm_path, title="Confusion Matrix")
print("Saved:", cm_path)
print(cm)

In [ ]:
#Evaluating ASVspoof 2021 DF evalution set:
model.eval()
all_preds_asv21 = []
all_labels_asv21 = []

with torch.no_grad():
    for inputs, labels in asv21_test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

        all_preds_asv21.extend(preds.cpu().numpy())
        all_labels_asv21.extend(labels.cpu().numpy())

# Metrics
acc = accuracy_score(all_labels_asv21, all_preds_asv21)
print(f"\nAccuracy: {acc:.4f}")

class_names = ["bonafide", "spoof"]
print("\nClassification Report:")
print(classification_report(all_labels_asv21, all_preds_asv21, target_names=class_names))


# Confusion Matrix
cm_path = model_folder+"Test_ASV21DF"
cm = save_cm_like_example(all_labels_asv21, all_preds_asv21, class_names, cm_path, title="Confusion Matrix")
print("Saved:", cm_path)
print(cm)